**Note: Run servers on local instead of Colab**

Here are some comments for a tutorial on Model serialization with Pickle/Joblib:

1. **Introduction to Model Serialization**:
   - Model serialization is the process of converting a machine learning model into a format that can be easily saved and loaded. This is crucial for deploying models in production environments or sharing them with others
.

2. **Why Use Pickle or Joblib?**:
   - Pickle and Joblib are two popular libraries in Python for serializing machine learning models. Pickle is a built-in Python module that can serialize and deserialize any Python object, including custom classes and objects
.
   - Joblib is more efficient than Pickle when working with large numpy arrays, making it a better choice for serializing large machine learning models
.

3. **Security Considerations**:
   - Both Pickle and Joblib use the pickle protocol under the hood, which can execute arbitrary code during unpickling. Therefore, it is important to only load models from trusted sources to avoid security risks
.

4. **Saving a Model with Pickle**:
   - To save a model using Pickle, you can use the `pickle.dump` function. This function serializes the model and writes it to a file. For example:
     ```python
     import pickle
     with open('model.pkl', 'wb') as file:
         pickle.dump(model, file)
     ```
   - This code opens a file in write-binary mode and uses Pickle to serialize the model into this file
.

5. **Loading a Model with Pickle**:
   - To load a model using Pickle, you can use the `pickle.load` function. This function reads the serialized model from a file and deserializes it. For example:
     ```python
     with open('model.pkl', 'rb') as file:
         model = pickle.load(file)
     ```
   - This code opens the file in read-binary mode and uses Pickle to deserialize the model from this file
.

6. **Saving a Model with Joblib**:
   - To save a model using Joblib, you can use the `joblib.dump` function. This function serializes the model and writes it to a file. For example:
     ```python
     from joblib import dump, load
     dump(model, 'model.joblib')
     ```
   - This code uses Joblib to serialize the model into a file named 'model.joblib'
.

7. **Loading a Model with Joblib**:
   - To load a model using Joblib, you can use the `joblib.load` function. This function reads the serialized model from a file and deserializes it. For example:
     ```python
     model = load('model.joblib')
     ```
   - This code uses Joblib to deserialize the model from the file named 'model.joblib'
.

8. **Performance Comparison**:
   - Joblib is generally faster than Pickle when dealing with large numpy arrays due to its optimizations for such data structures. However, Pickle may be faster for smaller models or models that do not contain large numpy arrays
.

9. **Use Cases**:
   - Use Pickle when you need to serialize small models or models that do not contain large numpy arrays. Use Joblib when you need to serialize large models or models that contain large numpy arrays
.

10. **Conclusion**:
    - Both Pickle and Joblib are powerful tools for model serialization. The choice between them depends on the size of your model and the data structures it contains. Always ensure that you load models from trusted sources to avoid security risks
.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])
pipeline.fit(X_train, y_train)


,steps,"[('scaler', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2


In [3]:
import pickle
version = '1.0.0'
filename_pkl = f'model_v{version}.pkl'
with open(filename_pkl, 'wb') as f:
    pickle.dump(pipeline, f)


In [4]:
import joblib
filename_joblib = f'model_v{version}.joblib'
joblib.dump(pipeline, filename_joblib)


['model_v1.0.0.joblib']

In [5]:
with open(filename_pkl, 'rb') as f:
    model_pickle = pickle.load(f)
preds_pickle = model_pickle.predict(X_test)


In [6]:
with open(filename_pkl, 'rb') as f:
    model_pickle = pickle.load(f)
preds_pickle = model_pickle.predict(X_test)


In [7]:
model_joblib = joblib.load(filename_joblib)
preds_joblib = model_joblib.predict(X_test)


In [8]:
from sklearn.metrics import accuracy_score
print(accuracy_score(y_test, preds_pickle))
print(accuracy_score(y_test, preds_joblib))


0.9649122807017544
0.9649122807017544


In [9]:
import os
versions = ['1.0.0', '1.1.0', '2.0.0']
for v in versions:
    fname = f'model_v{v}.joblib'
    joblib.dump(pipeline, fname)
os.listdir('.')


['.ipynb_checkpoints',
 'model_v1.0.0.joblib',
 'model_v1.0.0.pkl',
 'model_v1.1.0.joblib',
 'model_v2.0.0.joblib',
 'Week 4 - Day 3 Part2.ipynb',
 'Week_4_Day_3_Part1.ipynb']

In [10]:
import json
metadata = {
    'model_name': 'RandomForestPipeline',
    'versions': versions,
    'date_saved': {v: f'model_v{v}.joblib' for v in versions}
}
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f)


In [11]:
with open('model_metadata.json', 'r') as f:
    meta = json.load(f)
print(meta)


{'model_name': 'RandomForestPipeline', 'versions': ['1.0.0', '1.1.0', '2.0.0'], 'date_saved': {'1.0.0': 'model_v1.0.0.joblib', '1.1.0': 'model_v1.1.0.joblib', '2.0.0': 'model_v2.0.0.joblib'}}


Excercise

### Overview  
This exercise guides you through training a scikit-learn pipeline, saving it with both Pickle and Joblib, managing model versions and metadata, and exposing the model via a simple Flask API  ([9. Model persistence — scikit-learn 1.6.1 documentation](https://scikit-learn.org/stable/model_persistence.html?utm_source=chatgpt.com)).  

### Tasks  
1. **Data Preparation:** Load the Breast Cancer Wisconsin dataset (`sklearn.datasets.load_breast_cancer`), split into train/test (80/20), and standardize features  ([Save and Load Machine Learning Models in Python with scikit-learn](https://machinelearningmastery.com/save-load-machine-learning-models-python-scikit-learn/?utm_source=chatgpt.com)).  
2. **Pipeline Construction:** Build an `sklearn.pipeline.Pipeline` with `StandardScaler` and `RandomForestClassifier(n_estimators=100, random_state=42)` and fit it on the training data  ([How to properly pickle sklearn pipeline when using custom ...](https://stackoverflow.com/questions/57888291/how-to-properly-pickle-sklearn-pipeline-when-using-custom-transformer?utm_source=chatgpt.com)).  
3. **Serialization:**  
   - Save the fitted pipeline using Pickle protocol 5 to `model_v1.pkl`.  
   - Save the same pipeline using Joblib to `model_v1.joblib`  ([Save Machine Learning Model Using Pickle and Joblib](https://www.analyticsvidhya.com/blog/2021/08/quick-hacks-to-save-machine-learning-model-using-pickle-and-joblib/?utm_source=chatgpt.com)).  
4. **Version Control:** Simulate versioning by retraining the pipeline with a different random seed, then saving as `model_v2.pkl`/`.joblib`; repeat for a third version. Create a `model_metadata.json` cataloging versions, file names, and timestamps  ([Machine Learning Model Serialization - Christopher Flynn](https://flynn.gg/blog/machine-learning-model-serialization/?utm_source=chatgpt.com)).  
5. **Deserialization & Validation:**  
   - Load each Pickle and Joblib file, run `.predict()` on the test set, and report accuracy.  
   - Measure and compare file sizes and load times between Pickle and Joblib  ([Pickle is over 10 times faster than joblib for save and load scikit ...](https://www.reddit.com/r/Python/comments/ypj13g/pickle_is_over_10_times_faster_than_joblib_for/?utm_source=chatgpt.com)).  
6. **Model Serving (Bonus):**  
   - Implement a minimal Flask app with two endpoints:  
     - `GET /models` returns available version metadata.  
     - `POST /predict` accepts JSON feature vectors and returns predicted classes using the latest model.  
   - Demonstrate sending requests via `requests` in Colab  ([Model Serialization using pickle and joblib - Kaggle](https://www.kaggle.com/code/tasnimniger/model-serialization-using-pickle-and-joblib?utm_source=chatgpt.com)).  

### Deliverables  
- A Colab notebook containing all code cells above.  
- The saved model files (`.pkl`, `.joblib`) and `model_metadata.json`.  
- A brief report (markdown cell) comparing Pickle vs Joblib in speed and size.  
- (Optional) The Flask app code cell demonstrating model serving.

In [12]:
import time
from datetime import datetime
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

In [13]:
# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (455, 30) Test shape: (114, 30)


In [14]:
def build_and_train_pipeline(random_state=42):
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=random_state))
    ])
    pipeline.fit(X_train, y_train)
    return pipeline

# Version 1
model_v1 = build_and_train_pipeline(random_state=42)
acc_v1 = accuracy_score(y_test, model_v1.predict(X_test))
print("Model v1 accuracy:", acc_v1)

Model v1 accuracy: 0.956140350877193


In [15]:
# Save with Pickle (protocol 5)
with open('model_v1.pkl', 'wb') as f:
    pickle.dump(model_v1, f, protocol=5)

# Save with Joblib
joblib.dump(model_v1, 'model_v1.joblib')

print("Saved model_v1.pkl and model_v1.joblib")
print("Pickle size:", os.path.getsize('model_v1.pkl'), "bytes")
print("Joblib size:", os.path.getsize('model_v1.joblib'), "bytes")

Saved model_v1.pkl and model_v1.joblib
Pickle size: 314698 bytes
Joblib size: 325474 bytes


In [16]:
# Version 2 - different seed
model_v2 = build_and_train_pipeline(random_state=7)
acc_v2 = accuracy_score(y_test, model_v2.predict(X_test))

with open('model_v2.pkl', 'wb') as f:
    pickle.dump(model_v2, f, protocol=5)
joblib.dump(model_v2, 'model_v2.joblib')

# Version 3 - another seed
model_v3 = build_and_train_pipeline(random_state=99)
acc_v3 = accuracy_score(y_test, model_v3.predict(X_test))

with open('model_v3.pkl', 'wb') as f:
    pickle.dump(model_v3, f, protocol=5)
joblib.dump(model_v3, 'model_v3.joblib')

print("v2 accuracy:", acc_v2)
print("v3 accuracy:", acc_v3)

v2 accuracy: 0.956140350877193
v3 accuracy: 0.956140350877193


In [17]:
metadata = {
    "models": [
        {
            "version": "v1",
            "random_state": 42,
            "pickle_file": "model_v1.pkl",
            "joblib_file": "model_v1.joblib",
            "accuracy": acc_v1,
            "timestamp": datetime.now().isoformat()
        },
        {
            "version": "v2",
            "random_state": 7,
            "pickle_file": "model_v2.pkl",
            "joblib_file": "model_v2.joblib",
            "accuracy": acc_v2,
            "timestamp": datetime.now().isoformat()
        },
        {
            "version": "v3",
            "random_state": 99,
            "pickle_file": "model_v3.pkl",
            "joblib_file": "model_v3.joblib",
            "accuracy": acc_v3,
            "timestamp": datetime.now().isoformat()
        }
    ],
    "latest_version": "v3"
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

{
  "models": [
    {
      "version": "v1",
      "random_state": 42,
      "pickle_file": "model_v1.pkl",
      "joblib_file": "model_v1.joblib",
      "accuracy": 0.956140350877193,
      "timestamp": "2026-07-24T03:56:01.119884"
    },
    {
      "version": "v2",
      "random_state": 7,
      "pickle_file": "model_v2.pkl",
      "joblib_file": "model_v2.joblib",
      "accuracy": 0.956140350877193,
      "timestamp": "2026-07-24T03:56:01.119893"
    },
    {
      "version": "v3",
      "random_state": 99,
      "pickle_file": "model_v3.pkl",
      "joblib_file": "model_v3.joblib",
      "accuracy": 0.956140350877193,
      "timestamp": "2026-07-24T03:56:01.119896"
    }
  ],
  "latest_version": "v3"
}


In [18]:
results = []

for version in ["v1", "v2", "v3"]:
    pkl_path = f"model_{version}.pkl"
    joblib_path = f"model_{version}.joblib"

    # --- Pickle load ---
    start = time.time()
    with open(pkl_path, 'rb') as f:
        pkl_model = pickle.load(f)
    pkl_load_time = time.time() - start
    pkl_acc = accuracy_score(y_test, pkl_model.predict(X_test))
    pkl_size = os.path.getsize(pkl_path)

    # --- Joblib load ---
    start = time.time()
    jl_model = joblib.load(joblib_path)
    jl_load_time = time.time() - start
    jl_acc = accuracy_score(y_test, jl_model.predict(X_test))
    jl_size = os.path.getsize(joblib_path)

    results.append({
        "version": version,
        "pickle_load_time_s": round(pkl_load_time, 5),
        "pickle_accuracy": pkl_acc,
        "pickle_size_bytes": pkl_size,
        "joblib_load_time_s": round(jl_load_time, 5),
        "joblib_accuracy": jl_acc,
        "joblib_size_bytes": jl_size,
    })

for r in results:
    print(r)

{'version': 'v1', 'pickle_load_time_s': 0.04494, 'pickle_accuracy': 0.956140350877193, 'pickle_size_bytes': 314698, 'joblib_load_time_s': 0.0296, 'joblib_accuracy': 0.956140350877193, 'joblib_size_bytes': 325474}
{'version': 'v2', 'pickle_load_time_s': 0.02136, 'pickle_accuracy': 0.956140350877193, 'pickle_size_bytes': 313258, 'joblib_load_time_s': 0.02878, 'joblib_accuracy': 0.956140350877193, 'joblib_size_bytes': 324034}
{'version': 'v3', 'pickle_load_time_s': 0.01993, 'pickle_accuracy': 0.956140350877193, 'pickle_size_bytes': 308458, 'joblib_load_time_s': 0.02889, 'joblib_accuracy': 0.956140350877193, 'joblib_size_bytes': 319234}


In [ ]:
!pip install flask

In [ ]:
from flask import Flask, request, jsonify

app = Flask(__name__)

# Load metadata and the latest model at startup
with open('model_metadata.json', 'r') as f:
    METADATA = json.load(f)

latest_version = METADATA["latest_version"]
latest_model_path = f"model_{latest_version}.joblib"
LATEST_MODEL = joblib.load(latest_model_path)

@app.route('/models', methods=['GET'])
def get_models():
    return jsonify(METADATA)

@app.route('/predict', methods=['POST'])
def predict():
    payload = request.get_json()
    features = payload.get("features")

    if features is None:
        return jsonify({"error": "Missing 'features' key in JSON body"}), 400

    X_input = np.array(features).reshape(1, -1)
    prediction = LATEST_MODEL.predict(X_input)
    proba = LATEST_MODEL.predict_proba(X_input)

    return jsonify({
        "model_version": latest_version,
        "prediction": int(prediction[0]),
        "probabilities": proba[0].tolist()
    })

if __name__ == '__main__':
    app.run(port=5000)

In [ ]:
from threading import Thread

def run_app():
    app.run(port=5000, use_reloader=False)

Thread(target=run_app).start()

In [ ]:
import requests
import time

time.sleep(1)  # give Flask a moment to start

# Test GET /models
resp = requests.get("http://127.0.0.1:5000/models")
print("GET /models:", resp.json())

# Test POST /predict using a real test sample
sample = X_test[0].tolist()
resp = requests.post("http://127.0.0.1:5000/predict", json={"features": sample})
print("POST /predict:", resp.json())
print("Actual label:", y_test[0])

In [ ]:
from werkzeug.serving import make_server

class ServerThread(Thread):
    def __init__(self, app):
        super().__init__()
        self.server = make_server('127.0.0.1', 5000, app)
        self.ctx = app.app_context()
        self.ctx.push()

    def run(self):
        self.server.serve_forever()

    def shutdown(self):
        self.server.shutdown()

server_thread = ServerThread(app)
server_thread.start()
print("Server started")